# 01 Dataset Overview

Automatically discovers every CSV/TSV dataset in `ai/datasets`, loads each file safely, and prints a compact quality profile.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'ai' / 'datasets'
PREPROCESSING_DIR = PROJECT_ROOT / 'ai' / 'preprocessing'
MODEL_DIR = PROJECT_ROOT / 'ai' / 'models'

CSV_TSV_EXTENSIONS = {'.csv', '.tsv'}

def discover_data_files(data_dir=DATA_DIR):
    return sorted(p for p in data_dir.rglob('*') if p.suffix.lower() in CSV_TSV_EXTENSIONS)

def read_dataset(path):
    sep = '\t' if path.suffix.lower() == '.tsv' else ','
    return pd.read_csv(path, sep=sep)

def safe_name(path):
    return path.relative_to(DATA_DIR).as_posix().replace('/', '__')

def display_heading(text):
    try:
        from IPython.display import display, Markdown
        display(Markdown(f'### {text}'))
    except Exception:
        print(f'\n### {text}')


In [ ]:
files = discover_data_files()
print(f'Detected {len(files)} dataset files')
for p in files:
    print('-', p.relative_to(PROJECT_ROOT))


In [ ]:
for path in files:
    display_heading(path.relative_to(DATA_DIR).as_posix())
    try:
        df = read_dataset(path)
        print('Filename:', path.name)
        print('Shape:', df.shape)
        print('Columns:', list(df.columns))
        display(df.head())
        display_heading('Data types')
        display(df.dtypes.astype(str).to_frame('dtype'))
        display_heading('Missing values')
        display(df.isna().sum().to_frame('missing_values'))
        display_heading('Summary statistics')
        display(df.describe(include='all').transpose())
    except Exception as exc:
        print(f'Failed to load {path.name}: {exc}')
